# Predict Totals — Weekly Inference

Generates over/under predictions for upcoming NFL games and appends results
to `betting/totals_tracker.csv`.

**Tier logic:**
- `HIGH` = both XGBoost AND Ridge predict UNDER (both residuals < 0)
- `PASS` = everything else (no reliable OVER edge; model disagreement)

Run via papermill — same pattern as `predict_betting.ipynb`:
```bash
papermill betting/predict_totals.ipynb /tmp/out.ipynb -p MODE tuesday
```
Modes: `tuesday` (new predictions), `thursday` (injury refresh), `sunday` (final lock), `backfill`.

## Parameters

In [ ]:
MODE          = 'tuesday'   # papermill will override this
TARGET_SEASON = None        # None = auto-detect
TARGET_WEEK   = None        # None = auto-detect


## Imports + Model Load

In [ ]:
import warnings, json as _json
from pathlib import Path
import numpy as np
import pandas as pd
import nflreadpy as nfl
import joblib
warnings.filterwarnings('ignore')

# Resolve paths (works from project root or betting/)
_HERE = Path.cwd()
ALLPRO_CSV = next(
    (p for p in [_HERE / 'nfl_allpro_1997_2025.csv',
                 _HERE / 'betting/nfl_allpro_1997_2025.csv'] if p.exists()),
    _HERE / 'betting/nfl_allpro_1997_2025.csv'
)
WEATHER_CSV = next(
    (p for p in [_HERE / 'nfl_weather_2014_2025.csv',
                 _HERE / 'betting/nfl_weather_2014_2025.csv'] if p.exists()),
    _HERE / 'betting/nfl_weather_2014_2025.csv'
)
MODELS_DIR   = ALLPRO_CSV.parent / 'models'
TRACKER_PATH = ALLPRO_CSV.parent / 'totals_tracker.csv'

def _exec_notebook(path, glob):
    """Load all code cells from a notebook into glob namespace.
    Sets RUN_TESTS=False BEFORE executing so inline tests are skipped.
    """
    glob['RUN_TESTS'] = False
    with open(path, encoding='utf-8') as _fh:
        _nb = _json.load(_fh)
    for _cell in _nb['cells']:
        if _cell['cell_type'] == 'code':
            exec(''.join(_cell['source']), glob)

# 1. Load spread features from betting/features.py (single source of truth) — provides
#    build_features, PROD_FEATURES_35, TEAM_MAP, norm_name, etc. (was a json+exec of
#    features.ipynb; now an importable module tested by betting/test_features.py in CI).
import sys
_FEAT_DIR = next((p for p in [_HERE / 'betting', _HERE] if (p / 'features.py').exists()), None)
assert _FEAT_DIR, 'features.py not found; tried betting/ and CWD'
sys.path.insert(0, str(_FEAT_DIR))
import features as _features
globals().update({_k: _v for _k, _v in vars(_features).items() if not _k.startswith('__')})
assert 'build_features' in dir(), 'build_features not loaded from features.py'
print(f'Loaded features.py — PROD_FEATURES_35={len(PROD_FEATURES_35)}, build_features OK')

# 2. Load totals_features.ipynb — provides build_totals_features, TOTALS_FEATURE_COLS.
_TF_CANDIDATES = [_HERE / 'totals_features.ipynb', _HERE / 'betting/totals_features.ipynb']
_tf_path = next((p for p in _TF_CANDIDATES if p.exists()), None)
assert _tf_path, f'totals_features.ipynb not found; tried {_TF_CANDIDATES}'
_exec_notebook(_tf_path, globals())
print(f'Loaded totals_features.ipynb — TOTALS_FEATURE_COLS={len(TOTALS_FEATURE_COLS)}')

TOTALS_ALL_COLS = list(PROD_FEATURES_35) + TOTALS_FEATURE_COLS
assert len(TOTALS_ALL_COLS) == 49, f'Expected 49 total features, got {len(TOTALS_ALL_COLS)}'
print(f'Feature cols: {len(PROD_FEATURES_35)} spread + {len(TOTALS_FEATURE_COLS)} totals = {len(TOTALS_ALL_COLS)}')


## Load Models

In [ ]:
_xgb_pkl  = joblib.load(str(MODELS_DIR / 'totals_xgboost.pkl'))
_ridge_pkl = joblib.load(str(MODELS_DIR / 'totals_ridge.pkl'))
xgb_totals    = _xgb_pkl['model']
ridge_totals   = _ridge_pkl['model']
scaler_totals  = _ridge_pkl['scaler']
print(f'Loaded totals_xgboost.pkl  (train seasons: {_xgb_pkl["train_seasons"][0]}-{_xgb_pkl["train_seasons"][-1]})')
print(f'Loaded totals_ridge.pkl    (train seasons: {_ridge_pkl["train_seasons"][0]}-{_ridge_pkl["train_seasons"][-1]})')

# Hard-fail on feature order mismatch — a silent mismatch produces wrong predictions.
assert _xgb_pkl['feature_cols'] == TOTALS_ALL_COLS, (
    f'totals_xgboost.pkl feature_cols mismatch: pkl={len(_xgb_pkl["feature_cols"])} cols, '
    f'current={len(TOTALS_ALL_COLS)}. Retrain the pkl after any feature-list change.'
)
assert _ridge_pkl['feature_cols'] == TOTALS_ALL_COLS, (
    f'totals_ridge.pkl feature_cols mismatch: pkl={len(_ridge_pkl["feature_cols"])} cols, '
    f'current={len(TOTALS_ALL_COLS)}. Retrain the pkl after any feature-list change.'
)
print('Model load OK — feature order verified against pkls')


## Detect Week + Fetch Schedule

In [ ]:
import sys
_BETTING_CANDIDATES = [_HERE / 'predict_betting.ipynb',
                       _HERE / 'betting/predict_betting.ipynb']
_pb_path = next((p for p in _BETTING_CANDIDATES if p.exists()), None)
assert _pb_path, 'predict_betting.ipynb not found'
# get_week_info is defined in predict_betting cells 27-29
with open(_pb_path, encoding='utf-8') as _fh:
    _pb_nb = _json.load(_fh)
# Only exec cells that define get_week_info or norm_name (not the full pipeline)
for _i, _cell in enumerate(_pb_nb['cells']):
    if _cell['cell_type'] == 'code':
        _src = ''.join(_cell['source'])
        if 'def get_week_info' in _src or 'def _norm_name' in _src:
            exec(_src, globals())
del _pb_path, _pb_nb, _fh, _i, _cell, _src
assert callable(globals().get('get_week_info')), \
    'get_week_info not loaded from predict_betting.ipynb — has the helper signature changed?'

COACH_SEASONS = list(range(1999, 2026))
full_schedule = nfl.load_schedules(COACH_SEASONS).to_pandas()
if TARGET_SEASON is None:
    import datetime
    TARGET_SEASON = nfl.get_current_season()
TARGET_SEASON = int(TARGET_SEASON)
if TARGET_WEEK is None:
    TARGET_WEEK, _ = get_week_info(TARGET_SEASON, schedule_df=full_schedule)
    if TARGET_WEEK is None:
        raise ValueError('Season is over - no totals predictions to run.')
else:
    TARGET_WEEK = int(TARGET_WEEK)
print(f'Target: Season {TARGET_SEASON}, Week {TARGET_WEEK}, Mode {MODE}')


## Build Features for Upcoming Games

In [ ]:
# Upcoming REG-season games this week (sanity check)
upcoming_raw = full_schedule[
    (full_schedule['season'] == TARGET_SEASON) &
    (full_schedule['week']   == TARGET_WEEK) &
    (full_schedule['game_type'] == 'REG')
].copy()
if len(upcoming_raw) == 0:
    raise ValueError(f'No REG games found for {TARGET_SEASON} week {TARGET_WEEK}')
print(f'Upcoming games: {len(upcoming_raw)}')

# Prep inputs for build_features
allpro_df = pd.read_csv(ALLPRO_CSV)
allpro_df = allpro_df[allpro_df['Team'] != '2TM'].copy()
allpro_df['Team'] = allpro_df['Team'].replace(TEAM_MAP)

pbp_seasons = list(range(max(2014, TARGET_SEASON - 2), TARGET_SEASON + 1))
pbp_full_live = nfl.load_pbp(pbp_seasons).to_pandas()
# build_features expects pbp filtered to run/pass plays with both teams present
pbp_rp_live = pbp_full_live[
    pbp_full_live['play_type'].isin(['run', 'pass']) &
    pbp_full_live['posteam'].notna() &
    pbp_full_live['defteam'].notna()
].copy()

# Completed-games schedule for coach win pct + rolling SOS
coach_hist_df = full_schedule[full_schedule['result'].notna()].copy()

# Build the 35 spread features. KEYWORD ARGS REQUIRED — positional bindings
# don't match the build_features signature which is
# (target_week, target_season, full_schedule, pbp_rp, allpro_df, ...).
game_rows = build_features(
    target_week=TARGET_WEEK,
    target_season=TARGET_SEASON,
    full_schedule=full_schedule,
    pbp_rp=pbp_rp_live,
    allpro_df=allpro_df,
    coach_hist_df=coach_hist_df,
)
if game_rows is None or len(game_rows) == 0:
    raise ValueError(f'build_features returned no rows for {TARGET_SEASON} week {TARGET_WEEK}')
_missing_spread = [f for f in PROD_FEATURES_35 if f not in game_rows.columns]
assert not _missing_spread, f'build_features dropped spread features: {_missing_spread}'

# Add the 14 totals-specific features. Uses unfiltered pbp_full_live (pace
# computation counts ALL plays, not just run/pass).
game_rows = build_totals_features(
    game_rows, coach_hist_df, pbp_full_live, weather_path=WEATHER_CSV
)

print(f'Feature matrix: {len(game_rows)} games x {game_rows.shape[1]} cols')


## Run Inference

In [ ]:
# Fill missing feature cols with 0
for _c in TOTALS_ALL_COLS:
    if _c not in game_rows.columns:
        game_rows[_c] = 0.0

X_live = game_rows[TOTALS_ALL_COLS].fillna(0).values.astype('float32')

xgb_diff   = xgb_totals.predict(X_live)
ridge_diff  = ridge_totals.predict(scaler_totals.transform(X_live))

xgb_total   = game_rows['total_line'].values + xgb_diff
ridge_total = game_rows['total_line'].values + ridge_diff

# Tier: HIGH = both models predict UNDER
def get_tier(xgb_d, ridge_d):
    """HIGH when both models predict UNDER (residual < 0); PASS otherwise. No OVER bets."""
    if xgb_d < 0 and ridge_d < 0:
        return 'HIGH'
    return 'PASS'

tiers = [get_tier(x, r) for x, r in zip(xgb_diff, ridge_diff)]
print(f'HIGH (both UNDER): {tiers.count("HIGH")}  PASS: {tiers.count("PASS")}')


## Log Predictions

In [ ]:
import datetime

# Build a per-game actual-total lookup from full_schedule (for completed games).
# This way a re-run of an already-played week regrades the picks instead of
# wiping the grades.
_completed = full_schedule[full_schedule['home_score'].notna()].copy()
_completed['actual_total'] = _completed['home_score'] + _completed['away_score']
_actual_lkp = _completed.set_index('game_id')['actual_total'].to_dict()

rows = []
for i, row in game_rows.reset_index(drop=True).iterrows():
    gid       = row['game_id']
    tier      = tiers[i]
    line      = float(row['total_line'])
    actual    = _actual_lkp.get(gid)  # None if not yet played
    went_over = None
    model_correct = None
    if actual is not None and not pd.isna(actual):
        went_over = int(actual > line)
        if tier == 'HIGH':  # we always pick UNDER on HIGH; correct iff actual < line
            model_correct = int(actual < line)
        # PASS picks intentionally have model_correct=None (no bet → no grade).

    rows.append({
        'game_id':               gid,
        'home_team':             row['home_team'],
        'away_team':             row['away_team'],
        'gameday':               row['gameday'],
        'season':                TARGET_SEASON,
        'week':                  TARGET_WEEK,
        'total_line':            round(line, 1),
        'xgb_predicted_total':   round(float(xgb_total[i]), 1),
        'ridge_predicted_total': round(float(ridge_total[i]), 1),
        'xgb_diff':              round(float(xgb_diff[i]), 2),
        'ridge_diff':            round(float(ridge_diff[i]), 2),
        'consensus_tier':        tier,
        'recommendation':        'UNDER' if tier == 'HIGH' else 'PASS',
        'mode':                  MODE,
        'logged_at':             datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M'),
        'actual_total':          actual if actual is not None and not pd.isna(actual) else None,
        'went_over':             went_over,
        'model_correct':         model_correct,
    })

new_df = pd.DataFrame(rows)

# Upsert into tracker — preserve old grade columns when the new row has no grade.
# This protects against accidentally wiping grades on a re-run before the
# weekly grade-update step has caught up.
if TRACKER_PATH.exists():
    existing = pd.read_csv(TRACKER_PATH)
    # Bring forward old grades for any new row that's missing them
    if 'actual_total' in existing.columns:
        _existing_grades = existing.set_index('game_id')[['actual_total', 'went_over', 'model_correct']]
        for col in ['actual_total', 'went_over', 'model_correct']:
            _old_vals = new_df['game_id'].map(_existing_grades[col])
            new_df[col] = new_df[col].combine_first(_old_vals)
    existing = existing[~existing['game_id'].isin(new_df['game_id'])]
    combined = pd.concat([existing, new_df], ignore_index=True)
    combined = combined.drop_duplicates(subset=['game_id'], keep='last')
else:
    combined = new_df

combined = combined.sort_values(['season', 'week', 'game_id']).reset_index(drop=True)
combined.to_csv(TRACKER_PATH, index=False)

_high_n   = (new_df['consensus_tier'] == 'HIGH').sum()
_graded_n = new_df['model_correct'].notna().sum()
print(f'Wrote {len(new_df)} predictions to {TRACKER_PATH}')
print(f'  HIGH picks: {_high_n}  | graded: {_graded_n}')
print(f'Tracker total rows: {len(combined)}')

# Display summary
display_cols = ['home_team', 'away_team', 'total_line', 'xgb_predicted_total',
                'ridge_predicted_total', 'consensus_tier', 'recommendation',
                'actual_total', 'model_correct']
print(combined[combined['week'] == TARGET_WEEK][display_cols].to_string(index=False))
